# <center> Summarization using IndoT5 </center>

This notebook will show you how to finetuning T5 model on summarization task in Bahasa Indonesia. In this notebook, we will use [IndoSum](https://arxiv.org/abs/1810.05334) data, which is consist of news article and its summary. This notebook assume that you already download the data and put it in your google drive folder. Thus, you must let this notebook to have authorization for accessing your google drive (Don't worry it is safe).

## Install Dependencies

In [ ]:
# !pip install sentencepiece==0.1.95
# !pip install transformers==4.2.2
# !pip install datasets==1.2.0
# !pip install tqdm==4.48
# !pip install rouge

## Mount Google Drive

In [ ]:
# # uncomment this code if you run this on google colab and want to load the data from your gdrive 
# from google.colab import drive
# drive.mount('/content/drive')

# Import libraries

In [ ]:
import copy
import itertools
import evaluate
import torch
import contextlib
import math
from tqdm import tqdm
from torch.nn import CrossEntropyLoss
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, T5ForConditionalGeneration, DataCollatorForSeq2Seq
from transformers.tokenization_utils_base import BatchEncoding
from accelerate import Accelerator

# if there is an error related to tqdm, run this cell once more

## Data 

### Read data

In [ ]:
# I already download the data and put in this folder, you should change this depending on the location of the data in your drive
# In this practice, I use data from indosum dataset, you can go through this link to get the dataset https://github.com/indolem/indolem/tree/main/summarization_indosum
work_dir = "/content/drive/MyDrive/Summarization"
data_files = {"train": ['data/small.train.01.jsonl'], 
              "dev": ['data/small.dev.01.jsonl'], 
              "test": ['data/small.test.01.jsonl']
              }

# set streaming=True will create IterableDataset, good to save ram space since its not load all data into RAM
dataset = load_dataset('json', data_files=data_files, streaming=True)

train_dataset = dataset["train"]
dev_dataset = dataset["dev"]
test_dataset = dataset["test"]

In [ ]:
# check columns/features in the dataset
train_dataset.features

In [ ]:
# Lets, take a look on the firt teo of train dataset
for example in train_dataset:
    print(example)
    break

In [ ]:
# we use IndoT5-small from the model hub
tokenizer_checkpoint = "google/t5-efficient-mini" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_checkpoint)  

### Preprocess data

In [ ]:
# max encoder length
encoder_max_len = 512

In [ ]:
# encode function to preprocess the data
def encode(example, encoder_max_len: int=512) -> BatchEncoding:
    """Method for encoding text dataset

    Parameters
    ----------
    example : 
        Element of dataset obj.
    encoder_max_len : int, optional
        max token length, by default 512.

    Returns
    -------
    _type_
        _description_
    """

    # flatten the paragraph and summary data from dataset
    paragraph_wordpiece = ['summarize', ':' ] + list(itertools.chain.from_iterable(itertools.chain.from_iterable(example['paragraphs'])))
    summary_wordpiece = list(itertools.chain.from_iterable(example['summary']))

    # we need to put 'summarize: ' at the beginning of every paragraph, since that what the documentation tell to, you can change to another signature though
    encodings= tokenizer(text= paragraph_wordpiece, # the paragraph in dataset is in the form of list of sentence, and the sentence is in the form of list of words
                         text_target= summary_wordpiece, 
                         is_split_into_words=True, 
                         truncation=True, 
                         max_length= encoder_max_len,
                         padding=False)

    return encodings

In [ ]:
columns_remove = list(example.keys())
# preprocess/map the dataset using the encode function 

seed, buffer_size = 42, 10
train_dataset= train_dataset.map(encode, remove_columns=columns_remove)
train_dataset= train_dataset.shuffle(seed=seed, buffer_size=buffer_size)
dev_dataset= dev_dataset.map(encode, remove_columns=columns_remove)

In [ ]:
# Wrap data using dataloader since we will train the model by inputing the data batch by batch
# its not possible to input all data at once when training, since there is memory limitation on GPU 
batch_size = 4 

# set data collator for seq2seq, datacollator automatically convert encoding into pytorch tensor, don't need to convert encodings in dataset(train_ds, val_ds)
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer)

train_dl = DataLoader(train_dataset, batch_size=batch_size, collate_fn=data_collator)
dev_dl = DataLoader(dev_dataset, batch_size=batch_size, collate_fn=data_collator)

In [ ]:
for batch in dev_dl:
    break

{k: v.shape for k, v in batch.items() if k != 'offset_mapping'}

## Training phase

### Model preparation

We will train the model using gradient accumulation method, so we will update the model parameters after gathering the loss of few batchs (not a single data batch). 

In [ ]:
# Set accelerator
gradient_accumulation_step = 4 # number of steps before updating model parameters
accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_step)

In [ ]:
# load model

model_checkpoint = tokenizer_checkpoint
model = T5ForConditionalGeneration.from_pretrained(model_checkpoint)

# set optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

In [ ]:
# load model , opt, dataloader to optimizer, optional: you can also load scheduler
model, optimizer, train_dl, dev_dl = accelerator.prepare(model, optimizer, train_dl, dev_dl)


In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

In [ ]:
loss_fct = CrossEntropyLoss(ignore_index=-100, reduction='sum')

In [ ]:
def fit(num_epochs, num_train_batch_per_epoch, num_dev_batch_per_epoch, gradient_accumulation_step, model, train_dataset, train_loader, dev_loader, opt, loss_function):
    
    min_val_loss = 999
    training_iterator = itertools.cycle(train_loader)
    remainder = num_train_batch_per_epoch % gradient_accumulation_step
    remainder = remainder if remainder !=0 else gradient_accumulation_step
    total_train_updates = math.ceil(num_train_batch_per_epoch/gradient_accumulation_step)

    for epoch in range(num_epochs):

        # insert new seed using epoch
        train_dataset.set_epoch(epoch)
        model.train()
        total_train_loss = 0.0
        total_train_tokens = 0

        train_updates_pbar = tqdm(range(total_train_updates))
        for update_step in train_updates_pbar:
            if update_step == total_train_updates:
                break

            batch_samples = []
            num_batches_in_step = gradient_accumulation_step if update_step != (total_train_updates-1) else remainder
            for _ in range(num_batches_in_step):
                batch_samples += [next(training_iterator)]

            # get local num items in batch
            local_num_items_in_batch = sum([(batch["labels"].ne(-100)).sum() for batch in batch_samples])
            # to compute it correctly in a multi-device DDP training, we need to gather the total number of items in full batch
            num_items_in_batches = accelerator.gather(local_num_items_in_batch).sum().item()

            for i, batch in enumerate(batch_samples):
                if (i < len(batch_samples)-1 and accelerator.num_processes >1):
                    ctx = model.no_sync
                else:
                    ctx = contextlib.nullcontext

                with ctx():
                    input_ids, attention_mask, labels = batch["input_ids"], batch["attention_mask"], batch["labels"]
                    outputs = model(input_ids = input_ids, attention_mask = attention_mask, labels = labels)
                    loss = loss_function(outputs.logits.view(-1, outputs.logits.size(-1)), labels.view(-1))

                    loss = (loss * gradient_accumulation_step * accelerator.num_processes) / num_items_in_batches

                    accelerator.backward(loss)

            opt.step()
            opt.zero_grad()

            total_train_loss += (loss*num_items_in_batches)
            total_train_tokens += num_items_in_batches
            global_train_loss_avg = total_train_loss/total_train_tokens if total_train_tokens>0 else 0.0
            train_updates_pbar.set_description("(Epoch {}) TRAIN LOSS:{:.4f} LR:{:.8f}".format((epoch+1), global_train_loss_avg, get_lr(opt)))


        model.eval()
        pbar = tqdm(dev_loader, leave=True, total=num_dev_batch_per_epoch)
        with torch.no_grad():
            total_dev_loss = 0.0
            total_dev_tokens = 0
            for i, batch in enumerate(pbar):
                if i == num_dev_batch_per_epoch:
                    break
                input_ids, attention_mask, labels = batch["input_ids"], batch["attention_mask"], batch["labels"]
                
                outputs = model(input_ids = input_ids, attention_mask = attention_mask, labels = labels)
                dev_loss = loss_function(outputs.logits.view(-1, outputs.logits.size(-1)), labels.view(-1))

                local_dev_tokens = (batch["labels"].ne(-100)).sum()

                total_dev_loss += accelerator.gather_for_metrics(dev_loss).sum().item()
                total_dev_tokens += accelerator.gather_for_metrics(local_dev_tokens).sum().item()

                dev_loss_avg = total_dev_loss/total_dev_tokens
                pbar.set_description("(Epoch {}) DEV LOSS:{:.4f}".format((epoch+1), dev_loss_avg))

            global_dev_loss_avg = total_dev_loss/total_dev_tokens if total_dev_tokens > 0 else 0.0
            # we save model with the best val loss  
            if global_dev_loss_avg < min_val_loss:
                min_val_loss = global_dev_loss_avg
                model.save_pretrained("Results/best_model_summarization/")    

In [ ]:
# # Fit function without gradient accumulation and DDP setting

# def fit(num_epochs, num_train_batch_per_epoch, num_dev_batch_per_epoch, model, train_dataset, train_loader, dev_loader, opt):
    
#     min_val_loss = 999
#     for epoch in range(num_epochs):
#         # insert new seed using epoch
#         train_dataset.set_epoch(epoch)
#         model.train()
#         train_loss = 0
#         train_pbar = tqdm(train_loader, leave=True, total=num_train_batch_per_epoch)        
#         for i, batch_data in enumerate(train_pbar):
#             if i == num_train_batch_per_epoch:
#                 break
#             input_ids, attention_mask, labels = batch_data["input_ids"], batch_data["attention_mask"], batch_data["labels"]
#             input_ids = input_ids.to(device)
#             attention_mask = attention_mask.to(device)
#             labels = labels.to(device)
#             opt.zero_grad()
#             output = model(input_ids = input_ids, attention_mask = attention_mask, labels = labels)
#             loss = output.loss
#             train_loss += loss.item()
#             loss.backward()
#             opt.step()
#             train_loss_avg = train_loss/(i+1)
#             train_pbar.set_description("(Epoch {}) TRAIN LOSS:{:.4f} LR:{:.8f}".format((epoch+1), train_loss_avg, get_lr(opt)))

#         model.eval()
#         pbar = tqdm(dev_loader, leave=True, total=num_dev_batch_per_epoch)
#         with torch.no_grad():
#             val_loss = 0
#             for i, data in enumerate(pbar):
#                 if i == num_dev_batch_per_epoch:
#                     break
#                 input_ids, attention_mask, labels = data["input_ids"], data["attention_mask"], data["labels"]
#                 input_ids = input_ids.to(device)
#                 attention_mask = attention_mask.to(device)
#                 labels = labels.to(device)
#                 opt.zero_grad()
#                 output = model(input_ids = input_ids, attention_mask = attention_mask, labels = labels)
#                 loss = output.loss
#                 val_loss += loss.item()
#                 val_loss_avg = val_loss/(i+1)
#                 pbar.set_description("(Epoch {}) VALID LOSS:{:.4f}".format((epoch+1), val_loss_avg))

#             # we save model with the best val loss  
#             if val_loss_avg < min_val_loss:
#                 min_val_loss = val_loss_avg
#                 model.save_pretrained("Results/best_model_summarization/")    
                 

### Train model

In [ ]:
# number of training epoch
num_epochs = 3
num_train_batch_per_epoch = 100  # total data feeded into training per epoch is num_train_batch_per_epoch * batch_size
num_dev_batch_per_epoch = 25

In [ ]:
fit(num_epochs = num_epochs, 
    num_train_batch_per_epoch = num_train_batch_per_epoch , 
    num_dev_batch_per_epoch = num_dev_batch_per_epoch ,
    gradient_accumulation_step = gradient_accumulation_step,
    model = model,
    train_dataset= train_dataset, 
    train_loader = train_dl, 
    dev_loader = dev_dl, 
    opt = optimizer,
    loss_function=loss_fct
    )

## Testing phase

### Using the Model

In [ ]:
# get device
def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

In [ ]:
device = get_default_device()
print(device)

In [ ]:
best_model_checkpoint = "Results/best_model_summarization/"
best_model = T5ForConditionalGeneration.from_pretrained(best_model_checkpoint)
best_model = best_model.to(device)

In [ ]:
# this function will print the article, ist gold standard summary, and generated summary by the model for comparinson
def print_generated(sentence_text, summary_text, generated):
    
    b1 = "\033[1m"
    b2 = "\033[0m"
    for i in range(len(generated)):
        print(b1 + f"Full TEXT[{i}]: " + b2)
        print(sentence_text[i])
        print(b1 + f"Gold SUMMARY[{i}]: " + b2)
        print(summary_text[i])
        print(b1 + f"Generated SUMMARY[{i}]:" + b2)
        print(tokenizer.decode(generated[i], skip_special_tokens=True))
        print("\n")

In [ ]:
with torch.no_grad():
    data = next(iter(dev_dl))
    original_text = tokenizer.batch_decode(data['input_ids'], skip_special_tokens=True)
    summary_tokens = copy.deepcopy(data['labels'])
    summary_tokens[summary_tokens == -100]=tokenizer.pad_token_id
    summary_text = tokenizer.batch_decode(summary_tokens, skip_special_tokens=True)
    input_ids, attention_mask  = data["input_ids"], data["attention_mask"]
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    generated = best_model.generate(input_ids=input_ids, 
                               attention_mask=attention_mask, 
                               max_length=170, 
                               min_length=40, 
                               length_penalty=2.0, 
                               num_beams=4, 
                               early_stopping=True)

In [ ]:
print_generated(original_text, summary_text, generated)

*Note: HF pipeline is drop the support for "summarization" task for huggingface ver>5. To predict we need to create the predict function ourselves.* 

## Evaluation using test data

In [ ]:
# Lets valuate the model on test dataset, to save time, lets evaluate 20 of the test dataset. If you want to get more accurate evaluation score, its better to test it on more sample.
test_ds = test_dataset.map(encode, remove_columns = columns_remove)
test_dl = DataLoader(test_ds, batch_size=batch_size, collate_fn=data_collator)

In [ ]:
def pack_sentence_summary_generated(sentence_text, summary_text, generated):
    
    output = []
    for i in range(len(generated)):
        element = {}
        element['sentence'] = sentence_text[i]
        element['summary'] = summary_text[i]
        element['generated_summary'] = tokenizer.decode(generated[i], skip_special_tokens=True)
        output.append(element)

    return output

In [ ]:
def predict_summary(model, data_loader, num_batch = 100):
    
    output = []
    model.eval()
    with torch.no_grad():
        process_pbar = tqdm(data_loader, leave=True, total=num_batch)
        for i, batch_data in enumerate(process_pbar):

            if i>num_batch-1:
                break

            original_text = tokenizer.batch_decode(batch_data['input_ids'], skip_special_tokens=True)
            summary_tokens = copy.deepcopy(batch_data['labels'])
            summary_tokens[summary_tokens == -100]=tokenizer.pad_token_id
            summary_text = tokenizer.batch_decode(summary_tokens, skip_special_tokens=True)
            input_ids, attention_mask  = batch_data["input_ids"], batch_data["attention_mask"]
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            generated = model.generate(input_ids=input_ids, 
                                       attention_mask=attention_mask, 
                                       max_length=170, 
                                       min_length=40, 
                                       length_penalty=2.0, 
                                       num_beams=4, 
                                       early_stopping=True)
            
            # for this function will will include the generated by model summary and the gold standard summary, for easy of use later
            sub_output = pack_sentence_summary_generated(original_text, summary_text, generated)
            output += sub_output
            process_pbar.set_description("Progress")

            
    return output

In [ ]:
test_result = predict_summary(best_model, test_dl, num_batch = 20)

In [ ]:
len(test_result)

#### Evaluation using rouge score

In [ ]:
# using evaluate and rouge-score python package

rouge = evaluate.load("rouge")

In [ ]:
gold_summary = []
gen_summary = []

for item in test_result:
    gold_summary.append(item['summary'].replace('\r\n','').replace('\n',''))
    gen_summary.append(item['generated_summary'])

In [ ]:
result = rouge.compute(predictions=gen_summary, references=gold_summary, use_stemmer=True)
print({k: round(v, 4) for k,v in result.items()})

for benchmark purpose, check [this paper](https://arxiv.org/abs/2011.00677) .



## Download Model

In [ ]:
## Uncomment codes below to download your model

# !zip -r ./Results.zip ./Results

In [ ]:
# from google.colab import files

# files.download("./Results.zip")

**_author: Hadi Muhshi_** <br />
**_email: hadi.muhshi@gmail.com_**